# RAG Evaluation - issues with python dependencies 

In [1]:
import nest_asyncio
import qdrant_client

from llama_index.core import Settings
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore

from IPython.display import Markdown, display

In [2]:
import nest_asyncio

nest_asyncio.apply()

In [3]:
llm = Ollama(model="llama3.2:1b", request_timeout=120.0)

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-large-en-v1.5")

rerank = SentenceTransformerRerank(model="BAAI/bge-reranker-base", top_n=2)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
Settings.embed_model = embed_model

Settings.llm = llm


In [5]:
input_dir_path = './docs/paul_graham'

loader = SimpleDirectoryReader(
            input_dir = input_dir_path,
            required_exts=[".txt"],
            recursive=True
        )
docs = loader.load_data()


In [6]:
client = qdrant_client.QdrantClient(host="localhost", port=6333)

vector_store = QdrantVectorStore(client=client,
                                 collection_name="document_chat")

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(docs,
                                        storage_context=storage_context)

2026-10-08 09:55:11,432 - INFO - HTTP Request: GET http://localhost:6333/collections/document_chat/exists "HTTP/1.1 200 OK"
2026-10-08 09:55:11,433 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-10-08 09:55:11,435 - INFO - HTTP Request: GET http://localhost:6333/collections/document_chat "HTTP/1.1 200 OK"
2026-10-08 09:55:20,532 - INFO - HTTP Request: PUT http://localhost:6333/collections/document_chat/points?wait=true "HTTP/1.1 200 OK"


In [7]:
query_engine = index.as_query_engine(similarity_top_k=4,
                                     node_postprocessors=[rerank])

template = """Context information is below.
              ---------------------
              {context_str}
              ---------------------
              Given the context information above I want you to think
              step by step to answer the query in a crisp manner. Incase 
              you don't know the answer say 'I don't know!'.
              
              Query: {query_str}
              
              Answer:"""

qa_prompt_tmpl = PromptTemplate(template)

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)


2026-10-08 09:55:21,375 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [9]:
response = query_engine.query("""what did you work on before college?""")
                                 
display(Markdown(str(response)))

2026-10-08 09:56:00,140 - INFO - HTTP Request: POST http://localhost:6333/collections/document_chat/points/query "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-08 09:56:00,955 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Before college, you worked on two main things outside of school: writing and programming.